In [60]:
import tabml.datasets
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression


In [2]:
df = tabml.datasets.download_movielen_1m()
df.keys()

dict_keys(['users', 'movies', 'ratings'])

In [4]:
users, movies, ratings = df['users'], df['movies'], df['ratings']

In [6]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6040 entries, 0 to 6039
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   UserID      6040 non-null   int64 
 1   Gender      6040 non-null   object
 2   Age         6040 non-null   int64 
 3   Occupation  6040 non-null   int64 
 4   Zip-code    6040 non-null   object
dtypes: int64(3), object(2)
memory usage: 236.1+ KB


In [7]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3883 entries, 0 to 3882
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   MovieID  3883 non-null   int64 
 1   Title    3883 non-null   object
 2   Genres   3883 non-null   object
dtypes: int64(1), object(2)
memory usage: 91.1+ KB


In [8]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000209 entries, 0 to 1000208
Data columns (total 4 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   UserID     1000209 non-null  int64
 1   MovieID    1000209 non-null  int64
 2   Rating     1000209 non-null  int64
 3   Timestamp  1000209 non-null  int64
dtypes: int64(4)
memory usage: 30.5 MB


<h3>Create a movie feature matrix</h3>

In [11]:
genre_list = []
for genres in movies['Genres'].unique():
    for genre in genres.split('|'):
        if  genre not in genre_list:
            genre_list.append(genre)

genre_list

['Animation',
 "Children's",
 'Comedy',
 'Adventure',
 'Fantasy',
 'Romance',
 'Drama',
 'Action',
 'Crime',
 'Thriller',
 'Horror',
 'Sci-Fi',
 'Documentary',
 'War',
 'Musical',
 'Mystery',
 'Film-Noir',
 'Western']

In [15]:
movie_index_by_id = {id: i for i, id in enumerate(movies["MovieID"])}
genre_index_by_name = {name:i for i, name in enumerate(genre_list)}

movie_feature = np.zeros((len(movies), len(genre_list)))

for idx, movie_genres in enumerate(movies['Genres']):
    for genre in movie_genres.split('|'):
        genre_idx = genre_index_by_name[genre]
        movie_feature[idx, genre_idx] = 1

movie_feature

array([[1., 1., 1., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [24]:
movie_feature_ = np.c_[movies['Title'], movie_feature]
genre_list_ = ['Title']

for idx in genre_list:
    genre_list_.append(idx)

df_movie_feature = pd.DataFrame(data=movie_feature_, columns=genre_list_)
df_movie_feature

,Title,Animation,Children's,Comedy,Adventure,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Sci-Fi,Documentary,War,Musical,Mystery,Film-Noir,Western
0,Toy Story (1995),1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Jumanji (1995),0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Grumpier Old Men (1995),0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Waiting to Exhale (1995),0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Father of the Bride Part II (1995),0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3878,Meet the Parents (2000),0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3879,Requiem for a Dream (2000),0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3880,Tigerland (2000),0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3881,Two Family House (2000),0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


<h3>Training model</h3>

In [40]:
train_ratings, validation_ratings = train_test_split(
    ratings, test_size=0.2, random_state=42
)
print(train_ratings.head())
print('*'*43)
print(validation_ratings.head())

        UserID  MovieID  Rating  Timestamp
416292    2507     3035       2  974076680
683230    4087     2840       4  965431652
2434        19      457       3  978146863
688533    4118     2804       4  965804599
472584    2907      805       4  971838472
*******************************************
        UserID  MovieID  Rating  Timestamp
895536    5412     2683       2  960243649
899739    5440      904       5  959995181
55687      368     3717       4  976311423
63727      425     1721       4  976283587
822011    4942     3697       1  962642480


In [41]:
print('Number of users in training set: ', len(train_ratings['UserID'].unique()))
print('Number of users in validation set: ', len(validation_ratings['UserID'].unique()))
print('Total user: ', len(ratings['UserID'].unique()))

Number of users in training set:  6040
Number of users in validation set:  6038
Total user:  6040


In [65]:
def train_user_model(user_id):
    '''
    1. Take all rating data related to user_id.
    2. Take movies that the user_id has rated.
    3. Divide training set into 2 categories:
        3.1. Take all movies from movie_feature related to user_id as an independent set.
        3.2. Take all ratings of the user_id as a dependent set.
    4.  Using a model for training.
    '''
    user_ratings = train_ratings[train_ratings['UserID'] == user_id]
    movie_indexes = [
        movie_index_by_id[movie_id] for movie_id in user_ratings['MovieID']
    ]
    training_independent = movie_feature[movie_indexes]
    training_dependent = user_ratings['Rating']

    model = LinearRegression()
    model.fit(training_independent, training_dependent)
    return model

def predict(user_id, movie_id):
    ''' 
    1. Take the data in movie_feature related to the movie_id
    2. Predict the ratings of the user:
        2.1. Train the model related to the user_id.
        2.2. After training, the result is an object, using the predict function which is a built-in function of that object.
    '''
    movie_genre = movie_feature[movie_index_by_id[movie_id]].reshape((1, -1))
    predict_user = train_user_model(user_id).predict(movie_genre)
    return min(max(predict_user, 1), 5)

from sklearn.metrics import mean_squared_error

def compute_loss(ratings):
    predictions = np.zeros(len(ratings))
    for index, row in enumerate(ratings.itertuples(index=False)):
        predictions[index] = predict(row[0], row[1])
    rmse = mean_squared_error(ratings["Rating"], predictions, squared=False)
    return rmse
    

In [81]:
user_model_dict = {}
for user_id in users['UserID'].unique():
    user_model_dict[user_id] = train_user_model(user_id)

user_model_dict

AttributeError: 'dict' object has no attribute 'coef_'

In [80]:
def display(user_id, dict):
    user = dict[user_id].coef_
    dataFrame = {'Genre': genre_list, 'Predict': user}
    return pd.DataFrame(data=dataFrame)

display(301, user_model_dict)

,Genre,Predict
0,Animation,0.851336
1,Children's,-0.483542
2,Comedy,-0.313622
3,Adventure,-0.548298
4,Fantasy,0.215690
5,Romance,-0.285788
6,Drama,0.273217
7,Action,0.344347
8,Crime,-0.067644
9,Thriller,-0.080089
